<a href="https://colab.research.google.com/github/aagneye-syam/Ensemble-Based-Deep-Learning-Framework-for-Radiotherapy-Dose-Prediction/blob/master/3D_U_Net128_100epochs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Tue Jan  7 17:17:28 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   41C    P8               9W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
primary_directory = '/content/drive/My Drive/open-kbp-master'
sys.path.insert(0, primary_directory)

In [ ]:
# %tensorflow_version 2.x  # Uncomment if running in Google Colab to ensure TensorFlow 2.x
import shutil

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
import os
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
main_data_dir = '{}/provided-data'.format(primary_directory)
training_data_dir = '{}/train-val-pats'.format(main_data_dir)
validation_data_dir = '{}/validation-pats'.format(main_data_dir)
testing_data_dir = '{}/test-pats'.format(main_data_dir)

# Define hold out set
test_time = False  # Only change this to True when the model has been fully tuned on the validation set

# path where any data generated by this code (e.g., predictions, models) are stored
results_dir = '{}/results'.format(primary_directory)  # parent path where results are stored

In [ ]:
prediction_name = '3d_unet'
number_of_training_epochs = 100

In [ ]:
def load_file(file_name):
    """Load a file in one of the formats provided in the OpenKBP dataset
    :param file_name: the name of the file to be loaded
    :return: the file loaded
    """
    # Load the file as a csv
    loaded_file_df = pd.read_csv(file_name, index_col=0)
    # If the csv is voxel dimensions read it with numpy
    if 'voxel_dimensions.csv' in file_name:
        loaded_file = np.loadtxt(file_name)
    # Check if the data has any values
    elif loaded_file_df.isnull().values.any():
        # Then the data is a vector, which we assume is for a mask of ones
        loaded_file = np.array(loaded_file_df.index).squeeze()
    else:
        # Then the data is a matrix of indices and data points
        loaded_file = {'indices': np.array(loaded_file_df.index).squeeze(),
                       'data': np.array(loaded_file_df['data']).squeeze()}

    return loaded_file


def get_paths(directory_path, ext=''):
    """Get the paths of every file with a specified extension in a directory
    :param directory_path: the path of the directory of interest
    :param ext: the extensions of the files of interest
    :return: the path of all files of interest
    """
    # if dir_name doesn't exist return an empty array
    if not os.path.isdir(directory_path):
        return []
    # Otherwise dir_name exists and function returns contents name(s)
    else:
        all_image_paths = []
        # If no extension given, then get all files
        if ext == '':
            dir_list = os.listdir(directory_path)
            for iPath in dir_list:
                if '.' != iPath[0]:  # Ignore hidden files
                    all_image_paths.append('{}/{}'.format(directory_path, str(iPath)))
        else:
            # Get list of paths for files with the extension ext
            data_root = pathlib.Path(directory_path)
            for iPath in data_root.glob('*.{}'.format(ext)):
                all_image_paths.append(str(iPath))

    return all_image_paths


def get_paths_from_sub_directories(main_directory_path, sub_dir_list, ext=''):
    """Compiles a list of all paths within each sub directory listed in sub_dir_list that follows the main_dir_path
    :param main_directory_path: the path for the main directory of interest
    :param sub_dir_list: the name(s) of the directory of interest that are in the main_directory
    :param ext: the extension of the files of interest (in the usb directories)
    :return:
    """
    # Initialize list of paths
    path_list = []
    # Iterate through the sub directory names and build up the path list
    for sub_dir in sub_dir_list:
        paths_to_add = get_paths('{}/{}'.format(main_directory_path, sub_dir), ext=ext)
        path_list.extend(paths_to_add)

    return path_list


def sparse_vector_function(x, indices=None):
    """Convert a tensor into a dictionary of the non zero values and their corresponding indices
    :param x: the tensor or, if indices is not None, the values that belong at each index
    :param indices: the raveled indices of the tensor
    :return:  sparse vector in the form of a dictionary
    """
    if indices is None:
        y = {'data': x[x > 0], 'indices': np.nonzero(x.flatten())[-1]}
    else:
        y = {'data': x[x > 0], 'indices': indices[x > 0]}
    return y


def make_directory_and_return_path(dir_path):
    """Makes a directory only if it does not already exist
    :param dir_path: the path of the directory to be made
    :return: returns the directory path
    """
    os.makedirs(dir_path, exist_ok=True)

    return dir_path


def normalize_dicom(img):
    HOUNSFIELD_MIN = -1024
    HOUNSFIELD_MAX = 1500
    HOUNSFIELD_RANGE = 1000
    img[img < HOUNSFIELD_MIN] = HOUNSFIELD_MIN
    img[img > HOUNSFIELD_MAX] = HOUNSFIELD_MAX
    img = img / HOUNSFIELD_RANGE
    #if (HOUNSFIELD_RANGE!=0):
    #  img = ((img - HOUNSFIELD_MIN) / HOUNSFIELD_RANGE)
    #else:
    #  img = img - HOUNSFIELD_MIN
    return (img)


In [ ]:
class DataLoader:
    """Generates data for tensorflow"""

    def __init__(self, file_paths_list, batch_size=1, patient_shape=(128, 128, 128), shuffle=True,
                 mode_name='training_model'):
        """Initialize the DataLoader class, which loads the data for OpenKBP
        :param file_paths_list: list of the directories or single files where data for each patient is stored
        :param batch_size: the number of data points to lead in a single batch
        :param patient_shape: the shape of the patient data
        :param shuffle: whether or not order should be randomized
        """
        # Set file_loader specific attributes
        self.rois = dict(oars=['Brainstem', 'SpinalCord', 'RightParotid', 'LeftParotid',
                               'Esophagus', 'Larynx', 'Mandible'], targets=['PTV56', 'PTV63', 'PTV70'])

        self.batch_size = batch_size  # Number of patients to load in a single batch
        self.patient_shape = patient_shape  # Shape of the patient
        self.indices = np.arange(len(file_paths_list))  # Indices of file paths
        self.file_paths_list = file_paths_list  # List of file paths
        self.shuffle = shuffle  # Indicator as to whether or not data is shuffled
        self.full_roi_list = sum(map(list, self.rois.values()), [])  # make a list of all rois
        self.num_rois = len(self.full_roi_list)
        self.patient_id_list = ['pt_{}'.format(k.split('/pt_')[1].split('/')[0].split('.csv')[0]) for k in
                                self.file_paths_list]  # the list of patient ids with information in this data loader

        # Set files to be loaded
        self.required_files = None
        self.mode_name = mode_name  # Defines the mode for which data must be loaded for
        self.set_mode(self.mode_name)  # Set load mode to prediction by default

    def get_batch(self, index=None, patient_list=None):
        """Loads one batch of data
        :param index: the index of the batch to be loaded
        :param patient_list: the list of patients for which to load data for
        :return: a dictionary with the loaded data
        """

        if patient_list is None:
            # Generate batch based on the provided index
            indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        else:
            # Generate batch based on the request patients
            indices = self.patient_to_index(patient_list)

        # Make a list of files to be loaded
        file_paths_to_load = [self.file_paths_list[k] for k in indices]

        # Load the requested files as a tensors
        loaded_data = self.load_data(file_paths_to_load)
        return loaded_data

    def patient_to_index(self, patient_list):
        """Converts a list of patient ids to their appropriate indices
        :param patient_list: list of patient ids
        :return: list of indices for the requested patients
        """
        # Get the indices for the list that is not shuffled
        un_shuffled_indices = [self.patient_id_list.index(k) for k in patient_list]

        # Map the indices to the shuffled indices to the shuffled indices
        shuffled_indices = [self.indices[k] for k in un_shuffled_indices]

        return shuffled_indices

    def set_mode(self, mode_name, single_file_name=None):
        """Selects the type of data that is loaded
        :param mode_name: the name of the mode that the data loader is switching to
        :param single_file_name: the name of the file that should be loaded (only used if the mode_name is 'single_file')
        """
        self.mode_name = mode_name

        if mode_name == 'training_model':
            # The mode that should be used when training or validing a model
            self.required_files = {'dose': (self.patient_shape + (1,)),  # The shape of dose tensor
                                   'ct': (self.patient_shape + (1,)),  # The shape of ct tensor
                                   'structure_masks': (self.patient_shape + (self.num_rois,)),
                                   # The shape of the structure mask tensor
                                   'possible_dose_mask': (self.patient_shape + (1,)),
                                   # Mask of where dose can be deposited
                                   'voxel_dimensions': (3,)
                                   # Physical dimensions (in mm) of voxels
                                   }
        elif mode_name == 'dose_prediction':
            # The mode that should be used when training or validing a model
            self.required_files = {'ct': (self.patient_shape + (1,)),  # The shape of ct tensor
                                   'structure_masks': (self.patient_shape + (self.num_rois,)),
                                   # The shape of the structure mask tensor
                                   'possible_dose_mask': (self.patient_shape + (1,)),
                                   # Mask of where dose can be deposited
                                   'voxel_dimensions': (3,)  # Physical dimensions (in mm) of voxels
                                   }
            self.batch_size = 1
            print('Warning: Batch size has been changed to 1 for dose prediction mode')

        elif mode_name == 'predicted_dose':
            # This mode loads a single feature (e.g., dose, masks for all structures)
            self.required_files = {mode_name: (self.patient_shape + (1,))}  # The shape of a dose tensor

        elif mode_name == 'evaluation':
            # The mode that should be used evaluate the quality of predictions
            self.required_files = {'dose': (self.patient_shape + (1,)),  # The shape of dose tensor
                                   'structure_masks': (self.patient_shape + (self.num_rois,)),
                                   'voxel_dimensions': (3,),  # Physical dimensions (in mm) of voxels
                                   'possible_dose_mask': (self.patient_shape + (1,)),
                                   }
            self.batch_size = 1
            print('Warning: Batch size has been changed to 1 for evaluation mode')

        else:
            print('Mode does not exist. Please re-run with either \'training_model\', \'prediction\', '
                  '\'predicted_dose\', or \'evaluation\'')

    def number_of_batches(self):
        """Calculates how many full batches can be made in an epoch
        :return: the number of batches that can be loaded
        """
        return int(np.floor(len(self.file_paths_list) / self.batch_size))

    def on_epoch_end(self):
        """Randomizes the indices after each epoch"""
        if self.shuffle:
            np.random.shuffle(self.indices)

    def load_data(self, file_paths_to_load):
        """Generates data containing batch_size samples X : (n_samples, *dim, n_channels)
        :param file_paths_to_load: the paths of the files to be loaded
        :return: a dictionary of all the loaded files
        """

        # Initialize dictionary for loaded data and lists to track patient path and ids
        tf_data = {}.fromkeys(self.required_files)
        patient_list = []
        patient_path_list = []

        # Loop through each key in tf data to initialize the tensor with zeros
        for key in tf_data:
            # Make dictionary with appropriate data sizes for bath learning
            tf_data[key] = np.zeros((self.batch_size, *self.required_files[key]))

        # Generate data
        for i, pat_path in enumerate(file_paths_to_load):
            # Get patient ID and location of processed data to load
            patient_path_list.append(pat_path)
            pat_id = pat_path.split('/')[-1].split('.')[0]
            patient_list.append(pat_id)
            # Make a dictionary of all the tensors
            loaded_data_dict = self.load_and_shape_data(pat_path)
            # Iterate through the dictionary add the loaded data to the "batch channel"
            for key in tf_data:
                tf_data[key][i,] = loaded_data_dict[key]

        # Add two keys to the tf_data dictionary to track patient information
        tf_data['patient_list'] = patient_list
        tf_data['patient_path_list'] = patient_path_list

        return tf_data

    def load_and_shape_data(self, path_to_load):
        """ Reshapes data that is stored as vectors into matrices
        :param path_to_load: the path of the data that needs to be loaded. If the path is a directory, all data in the
         directory will be loaded. If path is a file then only that file will be loaded.
        :return: Loaded data with the appropriate shape
        """

        # Initialize the dictionary for the loaded files
        loaded_file = {}
        if '.csv' in path_to_load:
            loaded_file[self.mode_name] = load_file(path_to_load)
        else:
            files_to_load = get_paths(path_to_load, ext='')
            # Load files and get names without file extension or directory
            for f in files_to_load:
                f_name = f.split('/')[-1].split('.')[0]
                if f_name in self.required_files or f_name in self.full_roi_list:
                    loaded_file[f_name] = load_file(f)

        # Initialize matrices for features
        shaped_data = {}.fromkeys(self.required_files)
        for key in shaped_data:
            shaped_data[key] = np.zeros(self.required_files[key])

        # Populate matrices that were no initialized as []
        for key in shaped_data:
            if key == 'structure_masks':
                # Convert dictionary of masks into a tensor (necessary for tensorflow)
                for roi_idx, roi in enumerate(self.full_roi_list):
                    if roi in loaded_file.keys():
                        np.put(shaped_data[key], self.num_rois * loaded_file[roi] + roi_idx, int(1))
            elif key == 'possible_dose_mask':
                np.put(shaped_data[key], loaded_file[key], int(1))
            elif key == 'voxel_dimensions':
                shaped_data[key] = loaded_file[key]
            else:  # Files with shape
                np.put(shaped_data[key], loaded_file[key]['indices'], loaded_file[key]['data'])

        return shaped_data


In [ ]:
class My_generator(Ks.utils.Sequence):

  def __init__(self, data_load, num_files, batch_size):
    self.DataLoader = data_load
    self.num_files = num_files
    self.batch_size = batch_size

  def __len__(self):
    return (np.ceil(len(self.num_files)/float(self.batch_size))).astype(np.int)

  def __get_item__(self, id):
    #print(id)
    ct = []
    mask = []
    dose = []
    batch_data = self.DataLoader.get_batch(id)
    ct.append(normalize_dicom(batch_data['ct']))
    #ct.append(batch_data['ct'])
    mask.append(batch_data['structure_masks'])
    #dose.append(normalize_dicom(batch_data['dose']))
    dose.append(batch_data['dose'])
    #ct.append(batch_data['ct'])
    #dose.append(batch_data['dose'])
    ct = np.array(ct)
    mask = np.array(mask)
    dose = np.array(dose)
    #dose = dose / 70
    ct = ct.reshape(128,128,128,1)
    mask = mask.reshape(128,128,128,10)
    input = concatenate([ct,mask])
    dose = dose.reshape(128,128,128,1)
    #print(input.shape)
    #print(ct.shape)
    return input, dose


In [ ]:
batch_size = 1
train_val_split = 0.80
num_files = 240
#num_files = num_train + num_val
num_train = int(num_files * train_val_split)
num_val = num_files - num_train

training_plan_paths = get_paths(training_data_dir, ext='')
data_train = DataLoader(training_plan_paths)
train_gen = My_generator(data_train, num_train, batch_size)

val_plan_paths = get_paths(training_data_dir, ext='')
data_val = DataLoader(val_plan_paths)
val_gen = My_generator(data_val, num_val, batch_size)

test_plan_paths = get_paths(testing_data_dir, ext='')
num_test = len(test_plan_paths)
data_test = DataLoader(test_plan_paths)
test_gen = My_generator(data_test, num_test, batch_size)

In [ ]:
import tensorflow as tf # Import the tensorflow module and alias it as tf

seed = 2021

train_batch = 2
val_batch = 1
test_batch = 1
ds = tf.data.Dataset.range(num_files).shuffle(num_files, seed) # Now tf is defined and can be used
ds_train = ds.take(num_train)
ds_val = ds.skip(num_train)
ds.val = ds.take(num_val)
ds_train = ds_train.shuffle(num_train)
ds_train = ds_train.map(lambda x: tf.py_function(train_gen.__get_item__, [x], [tf.float32, tf.float32]), num_parallel_calls=tf.data.experimental.AUTOTUNE)
ds_val = ds_val.map(lambda x: tf.py_function(val_gen.__get_item__, [x], [tf.float32, tf.float32]), num_parallel_calls=tf.data.experimental.AUTOTUNE)
ds_train = ds_train.batch(train_batch)
ds_train = ds_train.prefetch(tf.data.experimental.AUTOTUNE)

ds_val = ds_val.batch(val_batch)
ds_val = ds_val.prefetch(tf.data.experimental.AUTOTUNE)

ds_test = ds.take(num_test)
ds_test = ds_test.map(lambda x: tf.py_function(test_gen.__get_item__, [x], [tf.float32, tf.float32]), num_parallel_calls=tf.data.experimental.AUTOTUNE)
ds_test = ds_test.batch(test_batch)
ds_test = ds_test.prefetch(tf.data.experimental.AUTOTUNE)

In [ ]:
from tensorflow.keras.layers import Input, LeakyReLU, BatchNormalization,Conv3D, concatenate, Activation, SpatialDropout3D, AveragePooling3D, Conv3DTranspose, MaxPooling3D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def conv_block(input, num_filters,kernel):
  conv = Conv3D(num_filters, kernel,  padding = 'same', kernel_initializer = 'he_normal')(input)
  conv = BatchNormalization()(conv)
  conv = Activation('relu')(conv)
  conv = Conv3D(num_filters, kernel,  padding = 'same', kernel_initializer = 'he_normal')(conv)
  conv = BatchNormalization()(conv)
  conv = Activation('relu')(conv)
  print(conv.shape)

  return conv

def get_u_net():
  p=0
  input_dim = (128,128,128,11)
  inputs = Ks.layers.Input(shape=input_dim, name="ct_mask")
  #ct_image = Input(128,128,128,1)
  #roi_masks = Input(self.roi_masks_shape)
  print(inputs.shape)
  #print(roi_masks.shape)
  #x = concatenate([ct, mask])
  x=inputs
  print(x.shape)

  print("Layer1")
  layer1 = conv_block(x,32,3)

  pool1 = MaxPooling3D(pool_size=(2, 2, 2),strides=(2,2,2),padding='same',data_format='channels_last')(layer1)
  drop1 = Dropout(p)(pool1)

  print("Layer 2")

  layer2 = conv_block(drop1,64,3)

  pool2 = MaxPooling3D(pool_size=(2, 2, 2),strides=(2,2,2),padding='same',data_format='channels_last')(layer2)
  drop2 = Dropout(p)(pool2)


  print("Layer 3")
  layer3 = conv_block(drop2,128,3)

  up4 = Conv3DTranspose(64,3, strides = (2, 2, 2), padding = 'same', kernel_initializer = 'he_normal')(layer3)
  up4 = concatenate([up4, layer2])
  print(up4.shape)
  drop4 = Dropout(p)(up4)

  print("layer 4")
  layer4 = conv_block(drop4,64,3)

  up5 = Conv3DTranspose(32,3, strides = (2, 2, 2), padding = 'same', kernel_initializer = 'he_normal')(layer4)
  up5 = concatenate([up5, layer1])
  drop5 = Dropout(p)(up5)


  print("Layer 5")
  layer5 = conv_block(drop5,32,3)

  final_dose = Conv3D(1, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(layer5)
  print(final_dose.shape)
  model = Model(inputs=[inputs], outputs=final_dose, name="generator")
  model.summary()
  return model



In [ ]:
dose_model = get_u_net()
opt = Adam(learning_rate=0.0001,decay=0.001,beta_1=0.5, beta_2=0.999)
#model.compile(loss =[dice_loss], optimizer=opt, metrics=[dice_coe])
dose_model.compile(loss ='mean_absolute_error', optimizer=opt)


In [ ]:
import datetime
from keras.callbacks import ModelCheckpoint,LearningRateScheduler, EarlyStopping,ReduceLROnPlateau

# Change the file extension to .keras
checkpoint = Ks.callbacks.ModelCheckpoint("/content/drive/My Drive/open-kbp/3D_UNet128_100epochs_valbest.keras", verbose=1,save_best_only=True)

logs_dir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tb_logs = Ks.callbacks.TensorBoard(log_dir=logs_dir)
#early_stopping = EarlyStopping(monitor='val_loss', min_delta = 0.0001, patience=30)
#adjust_learning_rate = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5, verbose=0, mode='auto', min_delta=0.0001, cooldown=0)
#callbacks = [checkpoint,early_stopping, adjust_learning_rate, tb_logs]
callbacks = [checkpoint, tb_logs]

In [ ]:
import tensorflow as tf # Import the tensorflow module and alias it as tf
import tensorflow.keras as Ks # Alias the Keras submodule of TensorFlow as Ks

seed = 2021

train_batch = 2
val_batch = 1
test_batch = 1
ds = tf.data.Dataset.range(num_files).shuffle(num_files, seed) # Now tf is defined and can be used
ds_train = ds.take(num_train)
ds_val = ds.skip(num_train)
ds.val = ds.take(num_val)
ds_train = ds_train.shuffle(num_train)
ds_train = ds_train.map(lambda x: tf.py_function(train_gen.__get_item__, [x], [tf.float32, tf.float32]), num_parallel_calls=tf.data.experimental.AUTOTUNE)
ds_val = ds_val.map(lambda x: tf.py_function(val_gen.__get_item__, [x], [tf.float32, tf.float32]), num_parallel_calls=tf.data.experimental.AUTOTUNE)

# Set the output shapes explicitly using set_shape
def set_shapes(input_data, target_data):
  input_data.set_shape([128, 128, 128, 11]) # Assuming this is the shape expected by your model
  target_data.set_shape([128, 128, 128, 1])  # Assuming this is the shape expected by your model
  return input_data, target_data

ds_train = ds_train.map(set_shapes)
ds_val = ds_val.map(set_shapes)

ds_train = ds_train.batch(train_batch)
ds_train = ds_train.prefetch(tf.data.experimental.AUTOTUNE)

ds_val = ds_val.batch(val_batch)
ds_val = ds_val.prefetch(tf.data.experimental.AUTOTUNE)

ds_test = ds.take(num_test)
ds_test = ds_test.map(lambda x: tf.py_function(test_gen.__get_item__, [x], [tf.float32, tf.float32]), num_parallel_calls=tf.data.experimental.AUTOTUNE)
ds_test = ds_test.map(set_shapes) # Apply set_shapes to ds_test as well
ds_test = ds_test.batch(test_batch)
ds_test = ds_test.prefetch(tf.data.experimental.AUTOTUNE)

In [ ]:
# ... (Previous code, including imports, data loading, model definition and compilation)...

# Fit the model and store the training history
history = dose_model.fit(ds_train, epochs=number_of_training_epochs, validation_data=ds_val, callbacks=callbacks)

# Plot the training and validation loss
fig = plt.figure(figsize=(6, 5))
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')  # Change legend labels
plt.savefig("memUnet100epochs")
plt.show()

# ... (Rest of your code)...

In [ ]:
mse_loss = dose_model.evaluate(ds_test)
print(mse_loss)

In [ ]:
dose_model = load_model("/content/drive/My Drive/open-kbp/3D_UNet128_100epochs.h5")
batch_size = 1
test_plan_paths = get_paths(testing_data_dir, ext='')
num_test = len(test_plan_paths)
data_test = DataLoader(test_plan_paths)
test_gen = My_generator(data_test, num_test, batch_size)

test_ct = []
test_dose=[]
test_mask=[]
input = []
batch_data=DataLoader.get_batch(data_test,10)
test_ct.append((normalize_dicom(batch_data['ct'])).reshape(128,128,128,1))
#test_ct.append(batch_data['ct'].reshape(128,128,128,1))
test_dose.append(batch_data['dose'].reshape(128,128,128,1))
test_mask.append(batch_data['structure_masks'].reshape(128,128,128,10))

plt.imshow(batch_data['ct'][0][55].reshape(128,128))
test_ct=np.array(test_ct)
test_mask=np.array(test_mask)
input = concatenate([test_ct,test_mask])


In [ ]:
pred =dose_model.predict(input)
print(pred.shape)
#pred = pred*70


In [ ]:
fig, ax = plt.subplots(nrows=1,ncols=2,figsize=(12,6))
#ax[0].imshow(test_ct[10].reshape(128,128))
#ax[0].set_title("CT")
ax[0].imshow(pred[0][55].reshape(128,128))
ax[0].set_title("Predicted")
#ax[1].imshow(normalize_dicom(batch_data['dose'][0][55]).reshape(128,128))
ax[1].imshow(batch_data['dose'][0][55].reshape(128,128))
ax[1].set_title("True")
plt.show()

In [ ]:
import tqdm
def predict_dose(data_test):
        """Predicts the dose for the given epoch number, this will only work if the batch size of the data loader
        is set to 1.
        :param epoch: The epoch that should be loaded to make predictions
        """
        # Define new models, or load most recent model if model already exists
        model = load_model("/content/drive/My Drive/open-kbp/3D_UNet128_100epochs.h5")
        os.makedirs('/content/drive/My Drive/open-kbp/results/3D_unet128_100epochs', exist_ok=True)
        # Use generator to predict dose


        number_of_batches = data_test.number_of_batches()
        print('Predicting dose')
        for idx in tqdm.tqdm(range(number_of_batches)):
            image_batch = data_test.get_batch(idx)

            # Get patient ID and make a prediction
            pat_id = image_batch['patient_list'][0]
            #input=concatenate([normalize_dicom(image_batch['ct'])])
            input=concatenate([normalize_dicom(image_batch['ct']), image_batch['structure_masks']])
            dose_pred_gy = model.predict(input)
            dose_pred_gy = dose_pred_gy * image_batch['possible_dose_mask']
            # Prepare the dose to save
            dose_pred_gy = np.squeeze(dose_pred_gy)
            dose_to_save = sparse_vector_function(dose_pred_gy)
            dose_df = pd.DataFrame(data=dose_to_save['data'].squeeze(), index=dose_to_save['indices'].squeeze(),
                                   columns=['data'])
            dose_df.to_csv('{}/{}.csv'.format('/content/drive/My Drive/open-kbp/results/3D_unet128_100epochs', pat_id))


In [ ]:
test_time = True
if test_time is False:
    hold_out_plan_paths = get_paths(validation_data_dir, ext='')  # list of paths used for held out validation
    stage_name = 'hold-out-validation'
else:
    hold_out_plan_paths = get_paths(testing_data_dir, ext='')  # list of paths used for held out testing
    stage_name = 'hold-out-testing'

In [ ]:
from provided_code.network_functions import PredictionModel
# Predict dose for the held out set
data_loader_hold_out = DataLoader(hold_out_plan_paths, mode_name='dose_prediction')
predict_dose(data_loader_hold_out)

In [ ]:
from itertools import product as it_product
class EvaluateDose:
    """Evaluate a full dose distribution against the reference dose on the OpenKBP competition metrics"""

    def __init__(self, data_loader, dose_loader=None):
        """
        Prepare the class for evaluating dose distributions
        :param data_loader: a data loader object that loads data from the reference dataset
        :param dose_loader: a data loader object that loads a dose tensor from any dataset (e.g., predictions)
        """
        # Initialize objects
        self.data_loader = data_loader  # Loads data related to ground truth patient information
        self.dose_loader = dose_loader  # Loads the data for a benchmark dose

        # Initialize objects for later
        self.patient_list = None
        self.roi_mask = None
        self.new_dose = None
        self.reference_dose = None
        self.voxel_size = None
        self.possible_dose_mask = None

        # Set metrics to be evaluated
        self.oar_eval_metrics = ['D_0.1_cc', 'mean']
        self.tar_eval_metrics = ['D_99', 'D_95', 'D_1']

        # Name metrics for data frame
        oar_metrics = list(it_product(self.oar_eval_metrics, self.data_loader.rois['oars']))
        target_metrics = list(it_product(self.tar_eval_metrics, self.data_loader.rois['targets']))

        # Make data frame to store dose metrics and the difference data frame
        self.metric_difference_df = pd.DataFrame(index=self.data_loader.patient_id_list,
                                                 columns=[*oar_metrics, *target_metrics])
        self.reference_dose_metric_df = self.metric_difference_df.copy()
        self.new_dose_metric_df = self.metric_difference_df.copy()

    def make_metrics(self):
        """Calculate a table of
        :return: the DVH score and dose score for the "new_dose" relative to the "reference_dose"
        """
        num_batches = self.data_loader.number_of_batches()
        dose_score_vec = np.zeros(num_batches)

        # Only make calculations if data_loader is not empty
        if not self.data_loader.file_paths_list:
            print('No patient information was given to calculate metrics')
        else:
            # Change batch size to 1
            self.data_loader.batch_size = 1  # Loads data related to ground truth patient information
            if self.dose_loader is not None:
                self.dose_loader.batch_size = 1  # Loads data related to ground truth patient information

            for idx in tqdm.tqdm(range(num_batches)):
                # Get roi masks for patient
                self.get_constant_patient_features(idx)
                # Get dose tensors for reference dose and evaluate criteria
                reference_dose = self.get_patient_dose_tensor(self.data_loader)
                if reference_dose is not None:
                    self.reference_dose_metric_df = self.calculate_metrics(self.reference_dose_metric_df, reference_dose)
                # If a dose loader was provided, calculate the score
                if self.dose_loader is not None:
                    new_dose = self.get_patient_dose_tensor(self.dose_loader)
                    # Make metric data frames
                    self.new_dose_metric_df = self.calculate_metrics(self.new_dose_metric_df, new_dose)
                    # Evaluate mean absolute error of 3D dose
                    dose_score_vec[idx] = np.sum(np.abs(reference_dose - new_dose)) / np.sum(self.possible_dose_mask)
                    # Save metrics at the patient level (this is a template for how DVH stream participants could save
                    # their files
                    #self.dose_metric_df.loc[self.patient_list[0]].to_csv('{}.csv'.format(self.patient_list[0]))

            if self.dose_loader is not None:
                dvh_score = np.nanmean(np.abs(self.reference_dose_metric_df - self.new_dose_metric_df).values)
                dose_score = dose_score_vec.mean()
                return dvh_score, dose_score
            else:
                print('No new dose provided. Metrics were only calculated for the provided dose.')

    def get_patient_dose_tensor(self, data_loader):
        """Retrieves a flattened dose tensor from the input data_loader.
        :param data_loader: a data loader that load a dose distribution
        :return: a flattened dose tensor
        """
        # Load the dose for the request patient
        dose_batch = data_loader.get_batch(patient_list=self.patient_list)
        dose_key = [key for key in dose_batch.keys() if 'dose' in key.lower()][0]  # The name of the dose
        dose_tensor = dose_batch[dose_key][0]  # Dose tensor
        return dose_tensor.flatten()

    def get_constant_patient_features(self, idx):
        """Gets the roi tensor
        :param idx: the index for the batch to be loaded
        """
        # Load the batch of roi mask
        rois_batch = self.data_loader.get_batch(idx)
        self.roi_mask = rois_batch['structure_masks'][0].astype(bool)
        # Save the patient list to keep track of the patient id
        self.patient_list = rois_batch['patient_list']
        # Get voxel size
        self.voxel_size = np.prod(rois_batch['voxel_dimensions'])
        # Get the possible dose mask
        self.possible_dose_mask = rois_batch['possible_dose_mask']


    def calculate_metrics(self, metric_df, dose):
        """
        Calculate the competition metrics
        :param metric_df: A DataFrame with columns indexed by the metric name and the structure name
        :param dose: the dose to be evaluated
        :return: the same metric_df that is input, but now with the metrics for the provided dose
        """
        # Prepare to iterate through all rois
        roi_exists = self.roi_mask.max(axis=(0, 1, 2))
        voxels_in_tenth_of_cc = np.maximum(1, np.round(100/self.voxel_size))  #
        for roi_idx, roi in enumerate(self.data_loader.full_roi_list):
            if roi_exists[roi_idx]:
                roi_mask = self.roi_mask[:, :, :, roi_idx].flatten()
                roi_dose = dose[roi_mask]
                roi_size = len(roi_dose)
                if roi in self.data_loader.rois['oars']:
                    if 'D_0.1_cc' in self.oar_eval_metrics:
                        # Find the fractional volume in 0.1cc to evaluate percentile
                        fractional_volume_to_evaluate = 100 - voxels_in_tenth_of_cc/roi_size * 100
                        metric_eval = np.percentile(roi_dose, fractional_volume_to_evaluate)
                        metric_df.at[self.patient_list[0], ('D_0.1_cc', roi)] = metric_eval
                    if 'mean' in self.oar_eval_metrics:
                        metric_eval = roi_dose.mean()
                        metric_df.at[self.patient_list[0], ('mean', roi)] = metric_eval
                elif roi in self.data_loader.rois['targets']:
                    if 'D_99' in self.tar_eval_metrics:
                        metric_eval = np.percentile(roi_dose, 1)
                        metric_df.at[self.patient_list[0], ('D_99', roi)] = metric_eval
                    if 'D_95' in self.tar_eval_metrics:
                        metric_eval = np.percentile(roi_dose, 5)
                        metric_df.at[self.patient_list[0], ('D_95', roi)] = metric_eval
                    if 'D_1' in self.tar_eval_metrics:
                        metric_eval = np.percentile(roi_dose, 99)
                        metric_df.at[self.patient_list[0], ('D_1', roi)] = metric_eval

        return metric_df


In [ ]:
# Evaluate dose metrics
data_loader_hold_out_eval = DataLoader(hold_out_plan_paths, mode_name='evaluation')  # Set data loader
prediction_paths = get_paths('/content/drive/My Drive/open-kbp/results/3D_unet128_100epochs', ext='csv')
hold_out_prediction_loader = DataLoader(prediction_paths, mode_name='predicted_dose')  # Set prediction loader
dose_evaluator = EvaluateDose(data_loader_hold_out_eval, hold_out_prediction_loader)
# print out scores if data was left for a hold out set
if not data_loader_hold_out_eval.file_paths_list:
    print('No patient information was given to calculate metrics')
else:
    dvh_score, dose_score = dose_evaluator.make_metrics()
    print('For this out-of-sample test:\n'
          '\tthe DVH score is {:.3f}\n '
          '\tthe dose score is {:.3f}'.format(dvh_score, dose_score))